# Minería de Datos · Semana 2 — sesión del sábado 15 de agosto

## Fuentes de datos e integración
### CSV · JSON · SQL · API — y por qué unirlos es donde se pierde la información

**26160 · grupo 020-81 · Sala de Informática 601**

---

En CRISP-DM esto es la **fase 2, comprensión de los datos**, y el comienzo de la **fase 3, preparación**.

La sesión del jueves terminó con una idea: *un resultado demasiado bueno es una alarma*. Hoy vamos por
la otra mitad de esa misma moneda: **un dato que llega mal integrado no produce una alarma. Produce un
modelo tranquilo y equivocado.**

> **Regla que se va a repetir cuatro veces hoy, hasta que moleste:**
> **cuente las filas antes y después de cada `merge`.** Si el número cambió y usted no lo esperaba,
> deténgase ahí. No siga.

## 0. Preparación

Nada que instalar: `pandas`, `json` y `sqlite3` vienen en Colab.

In [ ]:
import pandas as pd
import numpy as np
import json
import sqlite3

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

---

## 1. Las cuatro formas en que un dato llega a sus manos

| Forma | Cómo se ve | Quién la produce | El problema típico |
|---|---|---|---|
| **Archivo plano** (CSV, TXT) | Filas y columnas, texto | Una exportación de alguien | Separador, decimal, codificación |
| **JSON** | Anidado, en árbol | Una API o una app | Hay que **aplanarlo** antes de analizar |
| **Base relacional** (SQL) | Varias tablas con llaves | Un sistema en producción | Hay que **unir** las tablas |
| **API** | JSON por HTTP, paginado | Un servicio externo | Límites de consulta, y cambia sin avisar |

Su proyecto va a necesitar **por lo menos dos** de estas cuatro. Es el requisito 6 de la propuesta de
datasets: que haya algo que integrar.

### La celda de simulación

Las cuatro fuentes de hoy se fabrican aquí para que la clase corra sin internet.
**En su proyecto, cada una de estas celdas es un `read_csv`, un `read_json` o un `read_sql`.**
No hay que entender esta celda: hay que entender las que vienen después.

In [ ]:
rng = np.random.default_rng(7)
N = 400

# ── Fuente A: CSV de una secretaría (separador ';', decimal ',', y algo de mugre) ──
est = pd.DataFrame({
    "id_est":    [f"E{i:04d}" for i in range(1, N + 1)],
    "edad":      rng.integers(16, 30, N),
    "estrato":   rng.integers(1, 7, N),
    "promedio":  np.round(rng.normal(3.6, 0.5, N).clip(0, 5), 2),
    "localidad": rng.choice(["Usme", "Bosa", "Engativá", "Fontibón",
                             "Chapinero", "Ciudad Bolívar"], N),
})
# tres suciedades a propósito
est.loc[rng.choice(N, 25, replace=False), "promedio"] = np.nan
est.loc[5, "id_est"] = " E0006 "                      # espacios
est.loc[7, "id_est"] = "e0008"                        # minúscula
est_csv = est.copy()
est_csv["promedio"] = est_csv["promedio"].map(
    lambda v: "" if pd.isna(v) else str(v).replace(".", ","))
est_csv.to_csv("estudiantes.csv", sep=";", index=False, encoding="latin-1")

# ── Fuente B: JSON anidado, como lo devuelve una app ──
inscr = []
for i in range(1, N + 1):
    inscr.append({
        "estudiante": {"codigo": f"E{i:04d}", "programa": str(rng.choice(["Sistemas", "Industrial", "Catastral"]))},
        "matricula": {"semestre": int(rng.integers(1, 11)), "creditos": int(rng.integers(9, 22))},
        "contacto": {"correo": f"e{i:04d}@correo.co"},
    })
with open("inscripciones.json", "w", encoding="utf-8") as f:
    json.dump(inscr, f, ensure_ascii=False)

# ── Fuente C: base relacional con DOS tablas ──
con = sqlite3.connect("academico.db")
notas = pd.DataFrame({
    "codigo":    np.repeat([f"E{i:04d}" for i in range(1, N + 1)], 3),
    "id_asig":   rng.integers(1, 9, N * 3),
    "nota":      np.round(rng.normal(3.5, 0.8, N * 3).clip(0, 5), 1),
})
asign = pd.DataFrame({"id_asig": range(1, 9),
                      "asignatura": ["Cálculo", "Física", "Programación", "Bases de datos",
                                     "Redes", "Minería", "Ética", "Inglés"]})
notas.to_sql("notas", con, index=False, if_exists="replace")
asign.to_sql("asignaturas", con, index=False, if_exists="replace")

# ── Fuente D: "API" — un JSON paginado ──
api_paginas = [
    {"pagina": p, "siguiente": (p + 1 if p < 4 else None),
     "datos": [{"codigo": f"E{i:04d}", "beca": bool(rng.random() < 0.22)}
               for i in range(1 + (p - 1) * 100, 1 + p * 100)]}
    for p in range(1, 5)
]

print("Cuatro fuentes listas: estudiantes.csv · inscripciones.json · academico.db · api_paginas")

---

## 2. Fuente A — el CSV, y el error que no avisa

Primero se **mira** el archivo. Siempre. Antes de leerlo.

In [ ]:
with open("estudiantes.csv", "rb") as f:
    crudo = f.read(420)
print(crudo)

Tres cosas se ven ahí y las tres deciden cómo se importa:

1. El separador es `;`, no `,`.
2. El decimal es `,`, no `.`.
3. Aparecen bytes que no son ASCII —la `á` de «Engativá» sale como `\xe1`, no como `\xc3\xa1`—:
   **el archivo no está en UTF-8**, está en `latin-1`.

Ahora el error a propósito:

In [ ]:
mal = pd.read_csv("estudiantes.csv")     # sin decirle nada
print(mal.shape)
mal.head(3)

**¿Dio error?** No. Devolvió un DataFrame de **una sola columna** y siguió como si nada.

Esa es la lección: en preparación de datos, **la ausencia de un error rojo no es señal de que algo
salió bien.** Hay que verificar el `shape` y el `dtypes` siempre.

Ahora bien:

In [ ]:
est = pd.read_csv("estudiantes.csv", sep=";", decimal=",", encoding="latin-1")
print(est.shape)
print(est.dtypes)
est.head(3)

> **Pregunta 1.** `promedio` quedó como `float64` gracias a `decimal=","`. Si se hubiera olvidado ese
> argumento, ¿de qué tipo habría quedado, y qué habría pasado al calcular su media?

---

## 3. Fuente B — el JSON anidado

Un JSON no es una tabla: es un árbol. `json_normalize` lo aplana.

In [ ]:
with open("inscripciones.json", encoding="utf-8") as f:
    bruto = json.load(f)

print("tipo:", type(bruto), "| elementos:", len(bruto))
bruto[0]

In [ ]:
ins = pd.json_normalize(bruto)
print(ins.shape)
ins.head(3)

Fíjense en los nombres de columna: `estudiante.codigo`, `matricula.semestre`. `json_normalize` usa el
punto para marcar el nivel del que venía cada campo. Conviene renombrarlos antes de seguir.

In [ ]:
ins = ins.rename(columns={
    "estudiante.codigo":   "codigo",
    "estudiante.programa": "programa",
    "matricula.semestre":  "semestre",
    "matricula.creditos":  "creditos",
    "contacto.correo":     "correo",
})
ins.head(3)

---

## 4. Fuente C — la base relacional

Aquí la información **ya viene repartida en tablas**, y esa repartición es deliberada: es exactamente
lo que en Bases de Datos se llama normalización. Para analizar hay que volver a juntarla.

In [ ]:
con = sqlite3.connect("academico.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))

notas = pd.read_sql("SELECT * FROM notas", con)
asign = pd.read_sql("SELECT * FROM asignaturas", con)
print(notas.shape, asign.shape)
notas.head(3)

Se puede unir en Python… o dejar que el motor lo haga, que casi siempre es mejor:

In [ ]:
consulta = '''
SELECT n.codigo,
       a.asignatura,
       n.nota
FROM   notas n
JOIN   asignaturas a ON a.id_asig = n.id_asig
'''
notas_full = pd.read_sql(consulta, con)
notas_full.head(3)

> **Pregunta 2.** ¿Cuántas filas tiene `notas_full`? ¿Es una fila por estudiante? Si no lo es,
> **¿qué es una fila?** Esta pregunta decide todo lo que sigue.

In [ ]:
print("filas:", len(notas_full), "| estudiantes distintos:", notas_full["codigo"].nunique())

**Tres filas por estudiante.** Si esto se une sin más al resto, cada estudiante aparecerá tres veces.
Antes de integrar hay que **agregar** para llevarlo a una fila por estudiante:

In [ ]:
resumen_notas = (notas_full
                 .groupby("codigo")
                 .agg(nota_media=("nota", "mean"),
                      nota_min=("nota", "min"),
                      asignaturas=("asignatura", "count"))
                 .round(2)
                 .reset_index())
print(resumen_notas.shape)
resumen_notas.head(3)

---

## 5. Fuente D — la API paginada

Una API casi nunca devuelve todo de una vez: devuelve páginas. El patrón es siempre el mismo —pedir,
guardar, seguir al `siguiente` hasta que sea nulo—.

En un proyecto real esto sería:

```python
import requests
datos, url = [], "https://servicio.gov.co/api/becas?pagina=1"
while url:
    r = requests.get(url, timeout=30)
    r.raise_for_status()          # si falla, que falle aquí y no tres celdas después
    p = r.json()
    datos.extend(p["datos"])
    url = p["siguiente"]
```

Aquí las páginas ya están en memoria, pero el bucle es idéntico:

In [ ]:
datos, p = [], 1
while p is not None:
    pagina = api_paginas[p - 1]
    datos.extend(pagina["datos"])
    p = pagina["siguiente"]

becas = pd.DataFrame(datos)
print(becas.shape)
becas.head(3)

---

## 6. La integración — y la regla que no se negocia

Ya hay cuatro piezas. Ahora se unen. **Cada `merge` se hace contando filas antes y después.**

In [ ]:
def unir(izq, der, llave, como="left", nombre=""):
    "Merge que avisa cuando el numero de filas cambia sin permiso."
    antes = len(izq)
    out = izq.merge(der, on=llave, how=como)
    despues = len(out)
    señal = "OK" if despues == antes else ">>> OJO: CAMBIO EL NUMERO DE FILAS <<<"
    print(f"{nombre:<28} {antes:>6} -> {despues:>6}   {señal}")
    return out

In [ ]:
base = est.rename(columns={"id_est": "codigo"})
print("Partimos de", len(base), "estudiantes\n")

d1 = unir(base, ins,           "codigo", nombre="+ inscripciones (JSON)")
d2 = unir(d1,   resumen_notas, "codigo", nombre="+ notas (SQL, agregadas)")
d3 = unir(d2,   becas,         "codigo", nombre="+ becas (API)")

print("\nresultado:", d3.shape)
d3.head(3)

---

## 7. El desastre clásico: el `merge` que multiplica

¿Qué habría pasado si se une **sin agregar** las notas? Hágalo, y mire el número.

In [ ]:
_ = unir(base, notas_full[["codigo", "asignatura", "nota"]], "codigo",
         nombre="+ notas SIN agregar")

De 400 filas a 1.200. **Cada estudiante quedó triplicado.**

Y esto es lo grave: el modelo entrenado sobre esa tabla no falla. **Entrena tranquilo, y miente.**
Cada estudiante pesa tres veces, la validación cruzada mete al mismo estudiante en entrenamiento y en
prueba a la vez, y el acierto sale altísimo. Es un primo hermano de la fuga de información del jueves.

> **La regla, otra vez:** un `merge` de «uno a muchos» multiplica filas. Si usted quería una fila por
> estudiante, **agregue primero** y una después.

> **Pregunta 3.** ¿Por qué exactamente 1.200 y no otro número? ¿Y qué habría pasado si además la
> tabla de la derecha tuviera códigos repetidos por error?

---

## 8. Las llaves que no coinciden

El otro fallo silencioso: la llave existe en las dos tablas pero **no es idéntica**. El `merge`
no protesta: simplemente deja `NaN`.

In [ ]:
print("faltantes despues de integrar:")
print(d3[["programa", "nota_media", "beca"]].isna().sum())
print("\nSospechosos en la llave original:")
print(repr(base.loc[5, "codigo"]), "|", repr(base.loc[7, "codigo"]))

Dos filas quedaron sin pareja: una tenía espacios (`" E0006 "`) y la otra estaba en minúscula
(`"e0008"`). **Normalizar la llave antes de unir** resuelve la mayoría de estos casos:

In [ ]:
def normaliza_llave(s):
    return s.astype(str).str.strip().str.upper()

base_ok = base.copy()
base_ok["codigo"] = normaliza_llave(base_ok["codigo"])

d1 = unir(base_ok, ins,           "codigo", nombre="+ inscripciones")
d2 = unir(d1,      resumen_notas, "codigo", nombre="+ notas")
final = unir(d2,   becas,         "codigo", nombre="+ becas")

print("\nfaltantes ahora:")
print(final[["programa", "nota_media", "beca"]].isna().sum())

> **El tipo también cuenta.** Si una tabla trae la llave como `int64` y la otra como `object`, el
> `merge` no une **nada** y tampoco avisa. Verifíquelo siempre con `df.dtypes` antes de unir.

---

## 9. La revisión final, antes de dar la integración por buena

Cinco preguntas. Las cinco tienen que responderse **antes** de pasar a la fase 4.

In [ ]:
print("1. ¿Cuántas filas y columnas?      ", final.shape)
print("2. ¿Una fila por qué cosa?         ", "codigo únicos:", final["codigo"].nunique())
print("3. ¿Hay filas duplicadas?          ", final.duplicated().sum())
print("4. ¿Cuántos faltantes, y dónde?")
print(final.isna().sum()[lambda s: s > 0].sort_values(ascending=False))
print("\n5. ¿Los tipos son los correctos?")
print(final.dtypes)

In [ ]:
final.to_csv("integrado.csv", index=False, encoding="utf-8")
print("guardado: integrado.csv", final.shape)

> **Guarde siempre el crudo aparte.** El archivo integrado es un producto: se puede volver a generar.
> Los cuatro originales, no. Nunca sobreescriba una fuente.

---

## 10. Lo que hay que llevarse de hoy

1. **Mirar el archivo antes de leerlo.** Separador, decimal y codificación se deciden mirando bytes,
   no adivinando.
2. **Que no salga error no significa que salió bien.** Verifique `shape` y `dtypes` siempre.
3. **Un JSON es un árbol**: `json_normalize` lo aplana, y después hay que renombrar.
4. **Una base relacional viene repartida a propósito.** Reunirla es trabajo suyo, y conviene hacerlo
   en SQL.
5. **Cuente las filas antes y después de cada `merge`.** Si cambiaron sin que usted lo esperara,
   deténgase.
6. **Agregue antes de unir** cuando la relación es de uno a muchos.
7. **Normalice la llave** —`strip`, `upper`, tipo— antes de unir. Los `NaN` después de un merge casi
   siempre son un problema de llave, no de datos que falten.

---

## 11. Trabajo autónomo — entra en el taller semanal

**No hay entrega separada de esta clase.** Lo de hoy hace parte del **taller de la semana**, que se
entrega el **domingo 23 de agosto a las 11:59 p. m.** por Moodle, en un solo notebook.

Sobre **sus cuatro candidatos de datos** (los de la propuesta del 29 de agosto):

1. Para cada candidato: `shape`, `dtypes` y faltantes por columna.
2. Identifique **cuáles dos fuentes va a integrar** y **cuál es la llave** que las une.
3. Haga el `merge` usando la función `unir()` de la sección 6, y **pegue la salida**.
   Si el número de filas cambió, explique por qué en dos líneas.
4. Responda las tres preguntas numeradas de este notebook.

> Lo que se califica no es que el merge funcione: es que ustedes **se hayan dado cuenta** de cuándo no
> funcionó.